In [5]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

# ============================================================
# 1. SETTINGS
# ============================================================

seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

hidden_dim = 32
encoding_dim = 8
batch_size = 64
learning_rate = 1e-3
epochs = 100
patience = 10
validation_split = 0.2

# Manual thresholds selected after inspecting the reconstruction error histogram
threshold_list = [0.7]

results_root = Path("../results/AE_ECG5000_thresholds")
results_root.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LOAD ECG5000 DATASET
# ============================================================

train_df = pd.read_csv(
    "../data_raw/ECG5000/ECG5000_TRAIN.txt",
    sep=r"\s+",
    header=None
)

test_df = pd.read_csv(
    "../data_raw/ECG5000/ECG5000_TEST.txt",
    sep=r"\s+",
    header=None
)

y_train_orig = train_df.iloc[:, 0].values
X_train_all = train_df.iloc[:, 1:].values

y_test_orig = test_df.iloc[:, 0].values
X_test = test_df.iloc[:, 1:].values

# ============================================================
# 3. DEFINE NORMAL / ANOMALY CLASSES
# ============================================================

normal_class = 1

# 1 = normal, -1 = anomaly
y_test = np.where(y_test_orig == normal_class, 1, -1)

# ============================================================
# 4. TRAIN ONLY ON NORMAL DATA
# ============================================================

X_train_normal = X_train_all[y_train_orig == normal_class]

# ============================================================
# 5. STANDARDIZATION
# ============================================================

mu = X_train_normal.mean(axis=0)
sigma = X_train_normal.std(axis=0)
sigma[sigma == 0] = 1

X_train_normal_z = (X_train_normal - mu) / sigma
X_test_z = (X_test - mu) / sigma

# ============================================================
# 6. TRAIN FIXED AUTOENCODER
# ============================================================

input_dim = X_train_normal_z.shape[1]

def build_autoencoder(input_dim, hidden_dim, encoding_dim, learning_rate):
    inputs = Input(shape=(input_dim,))
    x = Dense(hidden_dim, activation="relu")(inputs)
    latent = Dense(encoding_dim, activation="relu")(x)
    x = Dense(hidden_dim, activation="relu")(latent)
    outputs = Dense(input_dim, activation="linear")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return model

model = build_autoencoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    encoding_dim=encoding_dim,
    learning_rate=learning_rate
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=patience,
    restore_best_weights=True
)

history = model.fit(
    X_train_normal_z,
    X_train_normal_z,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=validation_split,
    shuffle=True,
    callbacks=[early_stopping],
    verbose=0
)

train_recon = model.predict(X_train_normal_z, verbose=0)
train_errors = np.mean(np.square(X_train_normal_z - train_recon), axis=1)

test_recon = model.predict(X_test_z, verbose=0)
scores = np.mean(np.square(X_test_z - test_recon), axis=1)

print("Reconstruction error min:", scores.min())
print("Reconstruction error max:", scores.max())
print("Reconstruction error mean:", scores.mean())
print("Train reconstruction error min:", train_errors.min())
print("Train reconstruction error max:", train_errors.max())
print("Train reconstruction error mean:", train_errors.mean())

# ============================================================
# 7. EXPERIMENTS WITH MANUAL THRESHOLDS
# ============================================================

all_results = []

for threshold in threshold_list:
    exp_name = (
        f"hidden_{hidden_dim}_"
        f"latent_{encoding_dim}_"
        f"batch_{batch_size}_"
        f"lr_{learning_rate}_"
        f"threshold_{threshold}"
    )
    exp_name = exp_name.replace(".", "_").replace("-", "minus_")

    exp_dir = results_root / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    pred_custom = np.ones(len(scores))
    pred_custom[scores > threshold] = -1

    accuracy = accuracy_score(y_test, pred_custom) * 100
    precision = precision_score(y_test, pred_custom, pos_label=-1, zero_division=0) * 100
    recall = recall_score(y_test, pred_custom, pos_label=-1, zero_division=0) * 100
    f1 = f1_score(y_test, pred_custom, pos_label=-1, zero_division=0) * 100

    cm = confusion_matrix(y_test, pred_custom, labels=[-1, 1])

    result_row = {
        "experiment": exp_name,
        "hidden_dim": hidden_dim,
        "encoding_dim": encoding_dim,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "patience": patience,
        "epochs_trained": len(history.history["loss"]),
        "best_val_loss": round(float(np.min(history.history["val_loss"])), 6),
        "final_train_loss": round(float(history.history["loss"][-1]), 6),
        "threshold": threshold,
        "train_normal_samples": len(X_train_normal_z),
        "test_samples": len(X_test_z),
        "test_normal_samples": int((y_test == 1).sum()),
        "test_anomaly_samples": int((y_test == -1).sum()),
        "accuracy_%": round(accuracy, 2),
        "precision_%": round(precision, 2),
        "recall_%": round(recall, 2),
        "f1_%": round(f1, 2)
    }

    all_results.append(result_row)

    predictions_df = pd.DataFrame({
        "y_true": y_test,
        "pred_custom": pred_custom.astype(int),
        "reconstruction_error": scores,
        "threshold": threshold
    })

    predictions_df.to_csv(exp_dir / "predictions.csv", index=False)

    print("======================================")
    print("Autoencoder ECG5000")
    print("======================================")
    print(f"Threshold: {threshold}")
    print(f"Accuracy : {accuracy:.2f}%")
    print(f"Precision: {precision:.2f}%")
    print(f"Recall   : {recall:.2f}%")
    print(f"F1-score : {f1:.2f}%")
    print("\nConfusion Matrix")
    print("Rows = true labels [-1 anomaly, 1 normal]")
    print("Cols = predicted labels [-1 anomaly, 1 normal]")
    print(cm)

    # ========================================================
    # Metrics table
    # ========================================================

    metrics_df = pd.DataFrame({
        "Metrika": [
            "Presnosť (Accuracy)",
            "Precíznosť (Precision)",
            "Citlivosť (Recall)",
            "F1-skóre"
        ],
        "Hodnota (%)": [
            round(accuracy, 2),
            round(precision, 2),
            round(recall, 2),
            round(f1, 2)
        ]
    })

    fig, ax = plt.subplots(figsize=(6, 2))
    ax.axis("off")

    table = ax.table(
        cellText=metrics_df.values,
        colLabels=metrics_df.columns,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#dce6f1")
        else:
            cell.set_facecolor("#f9f9f9")

    plt.title(
        f"Výsledné metriky Autoencoder – ECG5000\nThreshold = {threshold}",
        fontsize=12,
        pad=20
    )

    plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ========================================================
    # Confusion matrix
    # ========================================================

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Matica zámen – Threshold = {threshold}")
    plt.colorbar()

    classes = ["Anomálie", "Normálne"]
    tick_marks = np.arange(len(classes))

    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    threshold_cm = cm.max() / 2.0 if cm.max() > 0 else 0.5

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "black" if cm[i, j] > threshold_cm else "white"
            plt.text(
                j, i, cm[i, j],
                ha="center",
                va="center",
                color=color
            )

    plt.xlabel("Predikované triedy")
    plt.ylabel("Skutočné triedy")
    plt.tight_layout()
    plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ========================================================
    # Reconstruction error histogram
    # ========================================================

    plt.figure(figsize=(10, 5))

    plt.hist(
        scores[y_test == 1],
        bins=50,
        alpha=0.7,
        label="Normálne"
    )

    plt.hist(
        scores[y_test == -1],
        bins=50,
        alpha=0.7,
        label="Anomálie"
    )

    plt.axvline(
        threshold,
        linestyle="--",
        linewidth=2,
        label=f"Threshold = {threshold}"
    )

    plt.title(f"Histogram rekonštrukčnej chyby Autoencoder – ECG5000\nThreshold = {threshold}")
    plt.xlabel("Rekonštrukčná chyba")
    plt.ylabel("Počet vzoriek")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(exp_dir / "reconstruction_error_histogram.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ========================================================
    # Example ECG reconstruction
    # ========================================================

    normal_idx = np.where(y_test == 1)[0][0]
    anom_idx = np.where(y_test == -1)[0][0]

    plt.figure(figsize=(10, 6))

    plt.subplot(2, 1, 1)
    plt.plot(X_test_z[normal_idx], label="Vstup")
    plt.plot(test_recon[normal_idx], label="Rekonštrukcia")
    plt.title("Rekonštrukcia normálneho EKG signálu")
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 1, 2)
    plt.plot(X_test_z[anom_idx], label="Vstup")
    plt.plot(test_recon[anom_idx], label="Rekonštrukcia")
    plt.title("Rekonštrukcia anomálneho EKG signálu")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(exp_dir / "ecg_reconstruction_examples.png", dpi=300, bbox_inches="tight")
    plt.close()

# ============================================================
# 8. SAVE SUMMARY
# ============================================================

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")


Reconstruction error min: 0.03427938723771543
Reconstruction error max: 25.723874524456797
Reconstruction error mean: 0.7049689241919845
Train reconstruction error min: 0.04415109103573052
Train reconstruction error max: 1.714062031368463
Train reconstruction error mean: 0.17283093233122657
Autoencoder ECG5000
Threshold: 0.7
Accuracy : 94.53%
Precision: 94.43%
Recall   : 92.31%
F1-score : 93.36%

Confusion Matrix
Rows = true labels [-1 anomaly, 1 normal]
Cols = predicted labels [-1 anomaly, 1 normal]
[[1729  144]
 [ 102 2525]]
Done
